In [1]:
from pyspark.sql import SparkSession

In [2]:
spark = SparkSession.builder.appName("DemoSparkApp").getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/16 03:38:21 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/06/16 03:38:34 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors


In [45]:
df1 = spark.read.csv("data/customers.csv")

In [5]:
# Customers
df1 = spark.read.csv(
    "data/customers.csv",
    header=True,
    inferSchema=True
)

# Order Items
df2 = spark.read.csv(
    "data/order_items.csv",
    header=True,
    inferSchema=True
)

# Orders
df3 = spark.read.csv(
    "data/orders.csv",
    header=True,
    inferSchema=True
)

# Products
df4 = spark.read.csv(
    "data/products.csv",
    header=True,
    inferSchema=True
)

# Returns
df5 = spark.read.csv(
    "data/returns.csv",
    header=True,
    inferSchema=True
)

In [47]:
result1 = df1.count()
result2 = df2.count()
result3 = df3.count()
result4 = df4.count()
result5 = df5.count()

In [48]:
print("total customer: ",result1)
print("total order_items: ",result2)
print("total orders: ",result3)
print("total products: ",result4)
print("total returns: ",result5)

total customer:  100000
total order_items:  3000000
total orders:  1000000
total products:  50000
total returns:  100000


In [49]:


result = spark.sql("""
SELECT
    category,
    SUM(unit_cost) AS total_sales
FROM df4
GROUP BY category
""")

result.show()
result.write.mode("overwrite").csv("output/df4.csv", header=True)

+--------------+------------------+
|      category|       total_sales|
+--------------+------------------+
|Home & Kitchen| 2901364.330000004|
|        Sports| 2853163.040000003|
|   Electronics|2864604.7399999946|
|      Clothing| 2841424.610000002|
|         Books|2853871.8500000075|
|        Beauty|2919388.7500000037|
|          Toys|2851913.1100000013|
+--------------+------------------+



In [6]:
df1.createOrReplaceTempView("df1")
df2.createOrReplaceTempView("df2")
df3.createOrReplaceTempView("df3")
df4.createOrReplaceTempView("df4")

result = spark.sql("""
SELECT
    c.customer_id,
    c.customer_name,
    SUM(oi.quantity * p.unit_cost) AS total_purchase
FROM df1 c
JOIN df3 o
ON c.customer_id = o.customer_id
JOIN df2 oi
ON o.order_id = oi.order_id
JOIN df4 p
ON oi.product_id = p.product_id
GROUP BY c.customer_id, c.customer_name
ORDER BY total_purchase DESC
LIMIT 10
""")

result.show()
result.write.mode("overwrite").csv("output/total_purchase.csv", header=True)

+-----------+--------------+------------------+
|customer_id| customer_name|    total_purchase|
+-----------+--------------+------------------+
|      93094|Customer_93094|          122887.1|
|      64560|Customer_64560|116660.08000000002|
|      23289|Customer_23289|109359.41999999998|
|      61218|Customer_61218|107647.11000000003|
|      40442|Customer_40442|104905.09000000001|
|      52275|Customer_52275|103149.83000000003|
|      52034|Customer_52034|102637.66999999998|
|      45631|Customer_45631|         102636.03|
|      84830|Customer_84830|102471.69999999998|
|      82593|Customer_82593|101936.83999999997|
+-----------+--------------+------------------+



In [13]:
result = spark.sql("""
SELECT
    MONTH(o.order_date) AS month,
    SUM(oi.quantity * p.unit_cost) AS total_sales
FROM df3 o
JOIN df2 oi
ON o.order_id = oi.order_id
JOIN df4 p
ON oi.product_id = p.product_id
WHERE YEAR(o.order_date) = (SELECT MAX(YEAR(order_date)) FROM df3)
GROUP BY MONTH(o.order_date)
ORDER BY month
""")

result.show()
result.write.mode("overwrite").csv("output/total_sales.csv", header=True)

+-----+--------------------+
|month|         total_sales|
+-----+--------------------+
|    1| 3.066959546199998E8|
|    2|2.8649646089000046E8|
|    3|3.0581029857000077E8|
|    4|2.9519848865999794E8|
|    5|3.0664839576999354E8|
|    6| 2.977815152800023E8|
|    7| 3.059577807799989E8|
|    8| 3.041606552800022E8|
|    9| 2.973250701699957E8|
|   10|  3.04292737079999E8|
|   11| 2.989098060799999E8|
|   12| 3.053784542299984E8|
+-----+--------------------+



In [22]:
df5.createOrReplaceTempView("df5")
result = spark.sql("""select category,
(COUNT(r.order_id) * 100.0 / COUNT(oi.order_id)) AS return_percentage
from df4 p
join df2 oi
on p.product_id = oi.product_id
left join df5 r
on oi.order_id = r.order_id
group by category
order by category""")

result.show()
result.write.mode("overwrite").csv("output/return_percentage.csv", header=True)

+--------------+-----------------+
|      category|return_percentage|
+--------------+-----------------+
|        Beauty|10.03235419129619|
|         Books|10.02350814590036|
|      Clothing| 9.97645033874562|
|   Electronics|10.00267670980709|
|Home & Kitchen|10.00336379177668|
|        Sports|10.02092306532332|
|          Toys|10.07903944537636|
+--------------+-----------------+



In [23]:
result = spark.sql("""select c.state,o.payment_mode,
COUNT(*) AS total
from df1 c
join df3 o
on c.customer_id = o.customer_id
group by c.state,o.payment_mode
ORDER BY c.state, total DESC""")
result.show()

[Stage 153:============================>                            (1 + 1) / 2]

+-----+----------------+-----+
|state|    payment_mode|total|
+-----+----------------+-----+
|   CA|             UPI|20246|
|   CA|     Net Banking|20054|
|   CA|      Debit Card|20024|
|   CA|Cash on Delivery|19999|
|   CA|     Credit Card|19979|
|   FL|      Debit Card|20010|
|   FL|     Credit Card|19888|
|   FL|Cash on Delivery|19762|
|   FL|             UPI|19740|
|   FL|     Net Banking|19629|
|   GA|     Net Banking|20041|
|   GA|Cash on Delivery|19771|
|   GA|      Debit Card|19733|
|   GA|     Credit Card|19722|
|   GA|             UPI|19608|
|   IL|Cash on Delivery|20498|
|   IL|     Net Banking|20404|
|   IL|     Credit Card|20361|
|   IL|             UPI|20359|
|   IL|      Debit Card|20178|
+-----+----------------+-----+
only showing top 20 rows



In [24]:
result = spark.sql("""
SELECT state, payment_mode, total
FROM (
    SELECT
        c.state,
        o.payment_mode,
        COUNT(*) AS total,
        ROW_NUMBER() OVER (
            PARTITION BY c.state
            ORDER BY COUNT(*) DESC
        ) AS rn
    FROM df1 c
    JOIN df3 o
    ON c.customer_id = o.customer_id
    GROUP BY c.state, o.payment_mode
)
WHERE rn = 1
""")

result.show()

[Stage 134:============================>                            (1 + 1) / 2]

+-----+----------------+-----+
|state|    payment_mode|total|
+-----+----------------+-----+
|   CA|             UPI|20246|
|   FL|      Debit Card|20010|
|   GA|     Net Banking|20041|
|   IL|Cash on Delivery|20498|
|   MI|      Debit Card|20416|
|   NC|     Net Banking|19596|
|   NY|      Debit Card|20369|
|   OH|     Net Banking|20351|
|   TX|             UPI|20065|
|   WA|             UPI|20244|
+-----+----------------+-----+



In [16]:
result = spark.sql("""select
p.category,SUM(oi.quantity * p.unit_cost) 
from df4 p
join df2 oi
on p.product_id = oi.product_id
group by p.category
order by p.category
limit 3
""")

result.show()

[Stage 53:=============================>                            (1 + 1) / 2]

+--------+---------------------------+
|category|sum((quantity * unit_cost))|
+--------+---------------------------+
|  Beauty|        5.259687848999895E8|
|   Books|       5.1482305433998823E8|
|Clothing|        5.117638649799961E8|
+--------+---------------------------+



In [17]:
result = spark.sql("""
SELECT category, product_name, revenue
FROM (
    SELECT
        p.category,
        p.product_name,
        SUM(oi.quantity * p.unit_cost) AS revenue,
        ROW_NUMBER() OVER (
            PARTITION BY p.category
            ORDER BY SUM(oi.quantity * p.unit_cost) DESC
        ) AS rn
    FROM df4 p
    JOIN df2 oi
    ON p.product_id = oi.product_id
    GROUP BY p.category, p.product_name
)
WHERE rn <= 3
ORDER BY category, revenue DESC
""")

result.show()

[Stage 59:>                                                         (0 + 2) / 2]

+--------------+-------------+------------------+
|      category| product_name|           revenue|
+--------------+-------------+------------------+
|        Beauty|Product_44016|189395.12000000002|
|        Beauty|Product_41537|188584.76999999996|
|        Beauty| Product_7205|188146.29000000004|
|         Books|Product_35314|         196801.74|
|         Books|Product_28311| 194764.8799999999|
|         Books|Product_27589| 193911.6800000001|
|      Clothing| Product_1560| 203042.1400000001|
|      Clothing| Product_7025|199598.28000000006|
|      Clothing|Product_31322|197628.41999999993|
|   Electronics| Product_6719| 203230.4599999999|
|   Electronics|Product_38170|         203011.26|
|   Electronics| Product_4836|194820.57999999996|
|Home & Kitchen| Product_5012|207483.03999999992|
|Home & Kitchen|Product_37452|          197124.9|
|Home & Kitchen|Product_27682|195609.96000000002|
|        Sports|Product_41700|204392.96000000008|
|        Sports|Product_32886|194810.87999999995|
